可換性×

In [39]:
import random
import numpy as np
import sys
from scipy.sparse import csr_matrix, hstack, vstack
import time
import pandas as pd

# --- インポートの一本化と表示設定 ---
np.set_printoptions(threshold=np.inf, linewidth=np.inf)
L=12
l_h = L // 2
J=3
P=768
np.set_printoptions(threshold=np.inf, linewidth=np.inf)

In [40]:
class FastAlgebraicOptimizer:
    def __init__(self, P=768, J=3, L_half=6):
        self.P = P
        self.J = J
        self.L_half = L_half
        self.L = 2 * L_half
        self.rho = self._generate_full_cycle(P)

    def _generate_full_cycle(self, P):
        indices = list(range(P))
        random.shuffle(indices)
        res = [0] * P
        for i in range(P):
            res[indices[i]] = indices[(i + 1) % P]
        return tuple(res)

    def get_perm_with_noise(self, exponent, has_swap=False):
        p = list(range(self.P))
        curr = list(self.rho)
        k = exponent % self.P
        while k > 0:
            if k % 2 == 1: p = [p[curr[i]] for i in range(self.P)]
            curr = [curr[curr[i]] for i in range(self.P)]
            k //= 2
        
        if has_swap:
            # 指定された位置にスワップを注入して非可換性を生む
            i1, i2 = random.sample(range(self.P), 2)
            p[i1], p[i2] = p[i2], p[i1]
        return tuple(p)

    def check_commute(self, p1, p2):
        for x in range(self.P):
            if p1[p2[x]] != p2[p1[x]]: return False
        return True

    def has_fixed_point(self, p1, p2, p3, p4):
        def inv(p):
            res = [0] * len(p)
            for i, v in enumerate(p): res[v] = i
            return res
        p2_inv, p4_inv = inv(p2), inv(p4)
        for x in range(self.P):
            if p4_inv[p3[p2_inv[p1[x]]]] == x: return True
        return False

    def solve(self):
        print(f"--- 非可換制約付き探索開始 (P={self.P}) ---")
        start_time = time.time()
        trial = 0
        while True:
            trial += 1
            f_exps = [random.randint(1, self.P - 1) for _ in range(self.L_half)]
            g_exps = [random.randint(1, self.P - 1) for _ in range(self.L_half)]
            
            # 指定されたインデックスにノイズを注入
            # f0, f1, g2, g3 にノイズを入れることで非可換ペアを作る
            F = [self.get_perm_with_noise(f_exps[i], has_swap=(i in [0, 1])) for i in range(self.L_half)]
            G = [self.get_perm_with_noise(g_exps[i], has_swap=(i in [2, 3])) for i in range(self.L_half)]

            # 条件：(f0, g3) および (f1, g2) が非可換であることを確認
            if self.check_commute(F[0], G[3]) or self.check_commute(F[1], G[2]):
                continue
            
            # 他の基本ペアが可換であることを確認（任意だがCSS条件への配慮）
            # ここではユーザー指定の非可換ペア以外は「可換になりやすい」生成をしている
            
            # C4 (4-cycle) のチェック
            def get_block(i, j):
                if j < self.L_half: return F[(j - i) % self.L_half]
                else: return G[(j - self.L_half - i) % self.L_half]

            is_clean = True
            for i in range(self.J):
                for ip in range(i + 1, self.J):
                    for j in range(self.L):
                        for jp in range(j + 1, self.L):
                            if self.has_fixed_point(get_block(i, j), get_block(ip, j), 
                                                    get_block(ip, jp), get_block(i, jp)):
                                is_clean = False; break
                        if not is_clean: break
                    if not is_clean: break
            
            if is_clean:
                duration = time.time() - start_time
                print(f"成功！ 試行回数: {trial}, 経過時間: {duration:.2f}秒")
                return F, G

In [41]:
# 実行
opt = FastAlgebraicOptimizer(P=768, J=3)
F_final, G_final = opt.solve()

--- 非可換制約付き探索開始 (P=768) ---
成功！ 試行回数: 1, 経過時間: 0.01秒


In [42]:

def tuple_to_sparse(p, size):
    rows = np.arange(size)
    cols = np.array(p)
    return csr_matrix((np.ones(size, dtype=np.int8), (rows, cols)), shape=(size, size))

def build_matrices_internal(F, G, P, J):
    L_h = len(F)
    F_m = [tuple_to_sparse(f, P) for f in F]
    G_m = [tuple_to_sparse(g, P) for g in G]
    hx_rows = []
    hz_rows = []
    for i in range(J):
        row_x = [F_m[(j-i)%L_h] for j in range(L_h)] + [G_m[(j-i)%L_h] for j in range(L_h)]
        row_z = [G_m[(i-j)%L_h].transpose() for j in range(L_h)] + [F_m[(i-j)%L_h].transpose() for j in range(L_h)]
        hx_rows.append(hstack(row_x))
        hz_rows.append(hstack(row_z))
    return vstack(hx_rows), vstack(hz_rows)

def count_cycles_direct(H):
    num_checks, num_vars = H.shape
    adj_c = [H.getrow(i).indices for i in range(num_checks)]
    H_csc = H.tocsc()
    adj_v = [H_csc.getcol(j).indices for j in range(num_vars)]
    c4, c6 = 0, 0
    # 4-cycle
    shared = {}
    for c1 in range(num_checks):
        for v in adj_c[c1]:
            for c2 in adj_v[v]:
                if c2 > c1:
                    shared[(c1, c2)] = shared.get((c1, c2), 0) + 1
    for count in shared.values():
        if count >= 2: c4 += count * (count - 1) // 2
    # 6-cycle
    for c1 in range(num_checks):
        for v1 in adj_c[c1]:
            for c2 in adj_v[v1]:
                if c2 <= c1: continue
                for v2 in adj_c[c2]:
                    if v2 == v1: continue
                    for c3 in adj_v[v2]:
                        if c3 <= c1 or c3 == c2: continue
                        common = set(adj_c[c3]) & set(adj_c[c1])
                        for v3 in common:
                            if v3 != v1 and v3 != v2: c6 += 1
    return c4, c6 // 2

In [43]:

# 最終確認
Hx, Hz = build_matrices_internal(F_final, G_final, P, J)
print(f"\nFinal Check - Hx Shape: {Hx.shape}")
# 直交性確認
ortho = (Hx @ Hz.transpose())
ortho.data %= 2
ortho.eliminate_zeros()
print(f"CSS Condition Violation: {ortho.nnz}")

c4, c6 = count_cycles_direct(Hx)
print(f"C4: {c4}, C6: {c6}")


Final Check - Hx Shape: (2304, 9216)
CSS Condition Violation: 288
C4: 0, C6: 9946


In [44]:
def display_fg_commutativity_table(F, G):
    size = len(F)
    f_labels = [f"f{i}" for i in range(size)]
    g_labels = [f"g{i}" for i in range(size)]
    matrix = np.zeros((size, size), dtype=int)
    
    for i in range(size):
        for j in range(size):
            if all(F[i][G[j][x]] == G[j][F[i][x]] for x in range(len(F[i]))):
                matrix[i, j] = 1
            else:
                matrix[i, j] = 0
    
    df = pd.DataFrame(matrix, index=f_labels, columns=g_labels)
    print("\n--- F-G 間可換表 (1=可換, 0=非可換) ---")
    return df

# 実行
# F_final, G_final = opt.solve() の実行後に以下を呼び出す
commute_df = display_fg_commutativity_table(F_final, G_final)
print(commute_df)


--- F-G 間可換表 (1=可換, 0=非可換) ---
    g0  g1  g2  g3  g4  g5
f0   0   0   0   0   0   0
f1   0   0   0   0   0   0
f2   1   1   0   0   1   1
f3   1   1   0   0   1   1
f4   1   1   0   0   1   1
f5   1   1   0   0   1   1
